在目前的 PyTorch 生态中，最著名的两大生命周期管理器是 **PyTorch Lightning** 和 **Hugging Face Accelerate**。

今天我们重点来拆解目前在学术界和中大型多模态项目（如各种 ViT、CLIP、SigLIP 复现）中用得最广、最能规范代码结构的 **PyTorch Lightning**。

---

## 一、 为什么要用 PyTorch Lightning？

传统的原生 PyTorch 训练代码，通常有两大痛点：

1. **“屎山”代码循环**：你必须自己手写 `for epoch in range`、`for batch in dataloader`、`optimizer.zero_grad()`、`loss.backward()`、`optimizer.step()`。这些代码在每个项目里有 90% 是完全重复的。
2. **硬件切换极其痛苦**：如果你想把单卡运行的代码，改成多卡并行（DDP）、或者改成混合精度训练（FP16/BF16）、亦或是移到 TPU 上跑，你需要修改大量的底层 DDP 初始化代码，极易写出 Bug。

**PyTorch Lightning 的核心哲学是：将“研究代码（模型、损失、优化器）”与“工程代码（训练循环、多卡同步、保存 Checkpoint）”彻底剥离。**

你只需要专注于写核心的模型逻辑，剩下的所有工程杂事，全部交由 Lightning 自动接管。

# PyTorch Lightning 完整学习笔记（上）

> 本笔记涵盖核心架构与上层 API，包括设计哲学、LightningModule、DataModule、Callback 系统和 Trainer 运行机制。  
> 所有关键代码均配有详细注释，帮助你理解**为什么这样写、每个参数的含义以及最佳实践**。

---

## 第一章：核心架构与设计哲学

### 1.1 Lightning 的本质

Lightning 是一个基于 **Hook（钩子）** 的深度学习运行时框架，它采用 **IoC（控制反转）** 思想：框架在合适的时机调用你预先定义好的函数。

### 1.2 Hook 思想

训练过程被拆分成一系列事件，每个事件对应一个你可以重写的 Hook：

```
on_train_start
→ on_train_epoch_start
  → on_train_batch_start
    → training_step
    → on_before_backward
    → backward
    → on_after_backward
    → on_before_optimizer_step
    → optimizer_step
  → on_train_batch_end
→ on_train_epoch_end
```

### 1.3 六大核心组件

```
Trainer （总调度器）
├── LightningModule  （模型 + 训练/验证/测试逻辑）
├── LightningDataModule （数据获取、预处理、拆分）
├── Callback （扩展逻辑：检查点、早停、EMA、监控等）
├── Strategy （分布式策略：DDP、FSDP、DeepSpeed）
├── Logger （实验记录：TensorBoard、WandB）
└── Precision Plugin （精度控制：FP32、FP16、BF16）
```

---

## 第二章：LightningModule 全生命周期 Hook

`LightningModule` 继承自 `torch.nn.Module`，同时承载模型定义、训练/验证/测试/预测逻辑以及优化器配置。

### 2.1 `__init__()` —— 模型与超参数初始化

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning.pytorch as pl

class LitModel(pl.LightningModule):
    def __init__(self, hidden_dim=128, lr=1e-3):
        """
        参数:
            hidden_dim: 隐藏层维度
            lr: 初始学习率
        """
        super().__init__()
        
        # ========== 必须：保存超参数到 self.hparams ==========
        # 所有传入 __init__ 的参数都会被自动保存，并在 checkpoint 中持久化。
        # 恢复训练或加载模型时，self.hparams 会被自动恢复。
        self.save_hyperparameters()
        
        # ========== 定义网络层 ==========
        self.backbone = nn.Sequential(
            nn.Linear(28 * 28, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 10)
        )
        
        # 在这里可以定义 loss 函数、torchmetrics 等
        # self.loss_fn = nn.CrossEntropyLoss()
```

### 2.2 `forward()` —— 纯推理前向传播

```python
    def forward(self, x):
        """
        纯推理方法，只包含从输入到输出的计算图。
        不要在这里计算 loss、记录日志或调用任何优化器相关操作。
        调用方式：logits = self(x)
        """
        # 展平图像后送入 backbone
        return self.backbone(x.view(x.size(0), -1))
```

### 2.3 `training_step()` —— 单步训练逻辑（最重要）

```python
    def training_step(self, batch, batch_idx):
        """
        每个训练 batch 调用一次。
        
        参数:
            batch: 来自 train_dataloader() 的一批数据，可以是 Tensor、tuple、dict 等
            batch_idx: 当前 epoch 中本 batch 的索引，从 0 开始
        
        返回:
            loss: 标量 Tensor 或包含 "loss" 键的字典。
                  Trainer 会自动提取 loss 并执行 backward() / optimizer.step() / zero_grad()。
        """
        x, y = batch                 # 解包数据
        logits = self(x)             # 调用 forward 得到预测值
        loss = F.cross_entropy(logits, y)  # 计算损失
        
        # ========== 日志记录 ==========
        # self.log 是 Lightning 统一的日志接口。
        # on_step=True: 每一步都记录（用于看板显示细粒度曲线）
        # on_epoch=True: 每个 epoch 结束时自动聚合（默认求平均）
        # prog_bar=True: 显示在进度条上
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        
        # ========== 返回 loss ==========
        return loss
```

### 2.4 `validation_step()` —— 验证逻辑

```python
    def validation_step(self, batch, batch_idx):
        """
        每个验证 batch 调用一次。
        Lightning 会自动设置 model.eval() 和 torch.no_grad()，你无需手动操作。
        
        多卡时必须设置 sync_dist=True，确保日志指标是全局统计而非单卡局部值。
        """
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        
        # 计算准确率
        preds = logits.argmax(dim=1)
        acc = (preds == y).float().mean()
        
        # 记录验证指标，同步多卡结果
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        self.log("val_acc", acc, prog_bar=True, sync_dist=True)
        
        # 可选：返回预测结果，用于在 on_validation_epoch_end 中做进一步处理
        return {"preds": preds, "targets": y}
```

### 2.5 `test_step()` 与 `predict_step()`

```python
    def test_step(self, batch, batch_idx):
        """
        测试阶段使用，与 validation_step 用法几乎相同。
        由 trainer.test() 触发。
        """
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        acc = (logits.argmax(1) == y).float().mean()
        self.log("test_loss", loss, sync_dist=True)
        self.log("test_acc", acc, sync_dist=True)

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        """
        生产环境推理使用。返回原始预测结果，Lightning 不记录日志或计算 loss。
        返回值会被收集成列表返回给用户。
        """
        x, _ = batch  # 假设只输入数据，忽略标签
        logits = self(x)
        return torch.softmax(logits, dim=-1)  # 返回概率
```

### 2.6 `configure_optimizers()` —— 配置优化器与调度器

```python
    def configure_optimizers(self):
        """
        返回优化器与学习率调度器。
        在 Trainer.fit() 开始前调用一次。
        """
        # 使用 self.hparams.lr 获取初始化时保存的学习率
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr)
        
        # 定义学习率调度器
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)
        
        # 返回方式1：仅返回优化器
        # return optimizer
        
        # 返回方式2：返回优化器和调度器（列表）
        # return [optimizer], [scheduler]
        
        # 返回方式3：使用字典精确控制调度器行为（推荐）
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",      # 按 epoch 还是 step 调整学习率
                "frequency": 1,           # 调整频率
                "monitor": "val_loss",    # 若使用 ReduceLROnPlateau，指定监控指标
            }
        }
```

### 2.7 训练生命周期钩子一览

以下钩子根据需要在你的 `LightningModule` 中重写，它们会在训练的不同阶段被 Trainer 自动调用。

```python
    def on_train_start(self):
        """训练开始前调用一次，可初始化训练专用资源。"""
        pass

    def on_train_epoch_start(self):
        """每个训练 epoch 开始时调用。可动态调整策略（如解冻层）。"""
        # 例：第10个 epoch 开始解冻 backbone
        # if self.current_epoch >= 10:
        #     for p in self.backbone.parameters():
        #         p.requires_grad = True
        pass

    def on_train_batch_start(self, batch, batch_idx):
        """每个训练 batch 开始前调用，可用于动态数据增强（Mixup、CutMix）。"""
        pass

    def on_before_backward(self, loss):
        """loss.backward() 之前调用，可检查 loss 是否有效。"""
        # 如果 loss 为 NaN，可以跳过本次反向传播（返回 -1）
        # if torch.isnan(loss):
        #     return -1
        pass

    def on_after_backward(self):
        """loss.backward() 之后调用，常用于梯度裁剪、记录梯度范数。"""
        # 梯度裁剪
        # torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=1.0)
        pass

    def on_before_optimizer_step(self, optimizer):
        """optimizer.step() 之前调用，可检查学习率、参数范数等。"""
        pass

    def optimizer_step(self, epoch, batch_idx, optimizer, optimizer_closure):
        """
        默认执行 optimizer.step(closure=optimizer_closure)。
        重写此方法可以完全控制优化器步进（如实现梯度累加、EMA 更新等）。
        仅在需要精细控制时重写。
        """
        # 例：每4个 batch 才更新一次（梯度累加）
        # if (batch_idx + 1) % 4 == 0:
        #     optimizer.step(closure=optimizer_closure)
        #     optimizer.zero_grad()
        optimizer.step(closure=optimizer_closure)

    def on_train_batch_end(self, outputs, batch, batch_idx):
        """一个 batch 训练完全结束后调用，outputs 是 training_step 的返回值。"""
        pass

    def on_train_epoch_end(self):
        """每个训练 epoch 结束时调用，可记录学习率等。"""
        # lr = self.trainer.optimizers[0].param_groups[0]['lr']
        # self.log("lr", lr)
        pass

    def on_train_end(self):
        """所有训练 epoch 结束后调用一次。"""
        pass
```

### 2.8 验证/测试/预测生命周期钩子

与训练钩子类似，前缀分别为 `on_validation_*`、`on_test_*`、`on_predict_*`，用法对称。

```python
    def on_validation_epoch_end(self):
        """所有验证 batch 结束后调用，可用于计算全局指标（如 F1、AUC）。"""
        # 例：从 self.trainer.callback_metrics 中获取 epoch 级指标
        pass
```

### 2.9 手动优化模式（Manual Optimization）

当训练流程复杂（GAN、RL、PPO）时，关闭自动优化，手写反向传播与参数更新。

```python
class ManualModel(pl.LightningModule):
    def __init__(self):
        super().__init__()
        self.automatic_optimization = False  # 关键：关闭自动优化
        
        self.generator = nn.Linear(100, 784)
        self.discriminator = nn.Linear(784, 1)

    def configure_optimizers(self):
        g_opt = torch.optim.Adam(self.generator.parameters(), lr=2e-4)
        d_opt = torch.optim.Adam(self.discriminator.parameters(), lr=2e-4)
        return [g_opt, d_opt]

    def training_step(self, batch, batch_idx):
        g_opt, d_opt = self.optimizers()  # 获取多个优化器
        
        # ---- 训练判别器 ----
        z = torch.randn(batch.size(0), 100, device=self.device)
        real = batch
        
        # 判别真实样本
        real_logits = self.discriminator(real)
        # 生成假样本（detach 切断梯度，防止影响生成器）
        fake = self.generator(z).detach()
        fake_logits = self.discriminator(fake)
        
        d_loss = F.binary_cross_entropy_with_logits(real_logits, torch.ones_like(real_logits)) + \
                 F.binary_cross_entropy_with_logits(fake_logits, torch.zeros_like(fake_logits))
        
        d_opt.zero_grad()
        self.manual_backward(d_loss)  # 必须使用 manual_backward，而不是 loss.backward()
        d_opt.step()
        
        # ---- 训练生成器 ----
        z = torch.randn(batch.size(0), 100, device=self.device)
        fake = self.generator(z)
        fake_logits = self.discriminator(fake)  # 注意这里不需要 detach，需要梯度流回 G
        
        g_loss = F.binary_cross_entropy_with_logits(fake_logits, torch.ones_like(fake_logits))
        
        g_opt.zero_grad()
        self.manual_backward(g_loss)
        g_opt.step()
        
        # 记录日志
        self.log("d_loss", d_loss, prog_bar=True)
        self.log("g_loss", g_loss, prog_bar=True)
```

---

## 第三章：LightningDataModule 全生命周期

### 3.1 一个完整的 DataModule 实现

```python
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchvision import transforms
import lightning.pytorch as pl

class MNISTDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = "./data", batch_size: int = 64):
        """
        仅保存配置，不做任何耗时操作（如下载、加载大数据）。
        """
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        
        # 定义数据预处理流水线（可以在 setup 中根据阶段调整）
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    def prepare_data(self):
        """
        仅主进程（global_rank 0）执行一次。
        用于下载数据、预训练模型、tokenizer 等，避免多卡并行时重复下载/冲突。
        不要在这里将数据集赋给 self，其他进程无法访问。
        """
        MNIST(self.data_dir, train=True, download=True)
        MNIST(self.data_dir, train=False, download=True)

    def setup(self, stage: str = None):
        """
        每个进程都会调用。根据 stage 构建不同的数据集，避免一次性加载所有数据。
        stage 可能值: 'fit', 'validate', 'test', 'predict' 或 None。
        """
        if stage == "fit" or stage is None:
            # 训练 + 验证阶段
            full_dataset = MNIST(self.data_dir, train=True, transform=self.transform)
            self.train_dataset, self.val_dataset = random_split(full_dataset, [55000, 5000])
        
        if stage == "test" or stage is None:
            # 测试阶段
            self.test_dataset = MNIST(self.data_dir, train=False, transform=self.transform)
        
        if stage == "predict":
            # 预测阶段
            self.predict_dataset = MNIST(self.data_dir, train=False, transform=self.transform)

    def train_dataloader(self):
        """返回训练 DataLoader，通常设置 shuffle=True。"""
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=4)

    def val_dataloader(self):
        """返回验证 DataLoader，通常设置 shuffle=False。"""
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def test_dataloader(self):
        """返回测试 DataLoader。"""
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def predict_dataloader(self):
        """返回预测 DataLoader。"""
        return DataLoader(self.predict_dataset, batch_size=self.batch_size, shuffle=False, num_workers=4)

    def teardown(self, stage: str = None):
        """阶段结束时释放资源（如关闭缓存、文件句柄）。"""
        pass
```

---

## 第四章：Callback 系统

### 4.1 内置 Callback 使用示例

```python
from lightning.pytorch.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    LearningRateMonitor,
    DeviceStatsMonitor,
    RichProgressBar,
)

# ---- ModelCheckpoint：自动保存最佳模型 ----
checkpoint_callback = ModelCheckpoint(
    monitor="val_loss",      # 监控的指标名，必须与 self.log 中的名称一致
    mode="min",              # 'min' 表示指标越小越好，'max' 表示越大越好
    save_top_k=3,            # 只保留最好的 3 个模型
    save_last=True,          # 同时保存最后一个 epoch 的模型（用于断点续训）
    dirpath="./checkpoints", # 保存目录
    filename="best-{epoch:02d}-{val_loss:.2f}"  # 自定义文件名
)

# ---- EarlyStopping：提前停止训练 ----
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,              # 连续 5 个验证 epoch 没有提升就停止
    mode="min"
)

# ---- LearningRateMonitor：自动记录学习率 ----
lr_monitor = LearningRateMonitor(logging_interval='epoch')

# ---- DeviceStatsMonitor：记录 GPU 利用率/显存 ----
device_stats = DeviceStatsMonitor()

# ---- RichProgressBar：美化进度条 ----
progress_bar = RichProgressBar()

# 将所有 Callback 传入 Trainer
trainer = pl.Trainer(
    callbacks=[
        checkpoint_callback,
        early_stop_callback,
        lr_monitor,
        device_stats,
        progress_bar
    ]
)
```

### 4.2 自定义 Callback 示例

```python
import lightning.pytorch as pl
import time

class TimerCallback(pl.Callback):
    """记录并打印训练总耗时。"""
    
    def on_fit_start(self, trainer, pl_module):
        # 训练开始时记录时间
        self.start_time = time.time()
        print("Training started...")
    
    def on_fit_end(self, trainer, pl_module):
        # 训练结束时计算并打印耗时
        duration = time.time() - self.start_time
        print(f"Training finished in {duration:.1f} seconds")
        
        # 也可以访问训练过程中的指标
        if "val_acc" in trainer.callback_metrics:
            final_acc = trainer.callback_metrics["val_acc"].item()
            print(f"Final validation accuracy: {final_acc:.4f}")
```

**Callback 执行顺序**：按 `Trainer(callbacks=...)` 列表中的顺序依次执行。通常先放 `ModelCheckpoint`，再放其他 Callback，确保先保存模型再执行上传等操作。

---

## 第五章：Trainer 全参数与运行机制

### 5.1 Trainer 配置示例（常见场景）

#### 1) 小模型实验 / 快速调试
```python
trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",   # 自动检测 CPU/GPU
    devices=1,
    fast_dev_run=False,   # 设为 True 可只跑 1 个 batch 检查管线
)
```

#### 2) 单卡 GPU 标准训练
```python
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed",     # 混合精度，节省显存并加速
    max_epochs=50,
    log_every_n_steps=50,
    callbacks=[checkpoint_callback, early_stop_callback]
)
```

#### 3) 多卡 DDP 训练
```python
trainer = pl.Trainer(
    accelerator="gpu",
    devices=4,               # 使用 4 张 GPU
    strategy="ddp",          # 分布式数据并行
    precision="bf16-mixed",  # BF16 更稳定（需要较新 GPU）
    gradient_clip_val=1.0,   # 梯度裁剪
    max_epochs=100,
)
```

#### 4) 大模型 FSDP + 梯度累积
```python
trainer = pl.Trainer(
    accelerator="gpu",
    devices=8,
    strategy="fsdp",              # 完全分片数据并行
    precision="bf16-mixed",
    gradient_clip_val=1.0,
    accumulate_grad_batches=8,    # 每 8 个 batch 更新一次权重，等效于大 batch
    max_epochs=3,
)
```

### 5.2 启动训练

```python
# 实例化模型和数据模块
model = LitModel(hidden_dim=256, lr=1e-3)
datamodule = MNISTDataModule(batch_size=128)

# 执行训练
trainer.fit(model, datamodule=datamodule)

# 测试
trainer.test(model, datamodule=datamodule)

# 预测
predictions = trainer.predict(model, datamodule=datamodule)
```

### 5.3 `self.log` 详解

```python
def training_step(self, batch, batch_idx):
    loss = ...
    # ========== self.log 参数说明 ==========
    self.log(
        "train_loss",          # 指标名称（字符串）
        loss,                  # 要记录的值（Tensor/float/int）
        prog_bar=True,         # 是否显示在进度条上
        logger=True,           # 是否发送给 Logger（TensorBoard/WandB）
        on_step=True,          # 是否记录每一步的值（训练时通常为 True）
        on_epoch=True,         # 是否在 epoch 结束时记录聚合值（验证时通常为 True）
        sync_dist=True,        # 多卡训练时是否同步所有卡的指标（验证时强烈推荐 True）
        reduce_fx="mean"       # epoch 聚合时的归约函数，默认求平均
    )
    return loss
```

**最佳实践**：
- 训练指标：`on_step=True, on_epoch=True`（同时保留细粒度和聚合曲线）
- 验证/测试指标：`on_step=False, on_epoch=True, sync_dist=True`
- 命名规范：使用 `/` 分隔层级，如 `"train/loss"`、`"val/acc"`，日志看板会自动分组。

---
